In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [2]:
df = pd.read_csv("../data/processed/featured_insurance_data.csv")
df.head()

,months_as_customer,age,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,insured_occupation,...,auto_model,auto_year,fraud_reported,policy_duration_days,claim_to_premium_ratio,injury_claim_ratio,property_claim_ratio,vehicle_claim_ratio,is_night_incident,multi_vehicle_flag
0,328,48,OH,250/500,1000,1406.91,0,MALE,MD,craft-repair,...,92x,2004,1,100,50.898778,0.090909,0.181818,0.727273,1,0
1,228,42,IN,250/500,2000,1197.22,5000000,MALE,MD,machine-op-inspct,...,E400,2007,1,3130,4.234811,0.153846,0.153846,0.692308,0,0
2,134,29,OH,100/300,2000,1413.14,5000000,FEMALE,PhD,sales,...,RAM,2007,0,5282,24.519864,0.222222,0.111111,0.666667,0,1
3,256,41,IL,250/500,2000,1415.74,6000000,FEMALE,PhD,armed-forces,...,Tahoe,2014,1,8996,44.782234,0.100000,0.100000,0.800000,1,0
4,228,44,IL,500/1000,1000,1583.91,6000000,MALE,Associate,sales,...,RSX,2009,0,256,4.103769,0.200000,0.100000,0.700000,0,0


In [3]:
print("Shape:", df.shape)
print("\nTarget distribution:\n")
print(df["fraud_reported"].value_counts())
print("\nTarget percentage:\n")
print(df["fraud_reported"].value_counts(normalize=True))

Shape: (1000, 41)

Target distribution:

fraud_reported
0    753
1    247
Name: count, dtype: int64

Target percentage:

fraud_reported
0    0.753
1    0.247
Name: proportion, dtype: float64


In [4]:
X = df.drop(columns=["fraud_reported"])
y = df["fraud_reported"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (800, 40)
X_test: (200, 40)


In [6]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 23
Categorical features: 17


In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight="balanced",
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [9]:
model_pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

In [10]:
results = []

for name, pipeline in model_pipelines.items():
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    if hasattr(pipeline, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_prob = None
    
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0)
    }
    
    if y_prob is not None:
        row["roc_auc"] = roc_auc_score(y_test, y_prob)
    
    results.append(row)

results_df = pd.DataFrame(results).sort_values(by="f1_score", ascending=False)
results_df

,model,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.825,0.625000,0.714286,0.666667,0.833356
2,Gradient Boosting,0.795,0.586957,0.551020,0.568421,0.838357
1,Random Forest,0.780,0.600000,0.306122,0.405405,0.840924


In [11]:
results_df.to_csv("../reports/model_results.csv", index=False)
print("Saved to ../reports/model_results.csv")

Saved to ../reports/model_results.csv


In [12]:
for name, pipeline in model_pipelines.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))

Logistic Regression
              precision    recall  f1-score   support

           0       0.90      0.86      0.88       151
           1       0.62      0.71      0.67        49

    accuracy                           0.82       200
   macro avg       0.76      0.79      0.77       200
weighted avg       0.83      0.82      0.83       200

Random Forest
              precision    recall  f1-score   support

           0       0.81      0.93      0.87       151
           1       0.60      0.31      0.41        49

    accuracy                           0.78       200
   macro avg       0.70      0.62      0.64       200
weighted avg       0.76      0.78      0.75       200

Gradient Boosting
              precision    recall  f1-score   support

           0       0.86      0.87      0.87       151
           1       0.59      0.55      0.57        49

    accuracy                           0.80       200
   macro avg       0.72      0.71      0.72       200
weighted avg       0.7

In [13]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = model_pipelines[best_model_name]

print("Best model:", best_model_name)

Best model: Logistic Regression


In [14]:
best_pipeline.fit(X_train, y_train)
risk_scores = best_pipeline.predict_proba(X_test)[:, 1]

evaluation_df = X_test.copy()
evaluation_df["actual_fraud"] = y_test.values
evaluation_df["risk_score"] = risk_scores
evaluation_df = evaluation_df.sort_values(by="risk_score", ascending=False)

evaluation_df.head(10)

,months_as_customer,age,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,insured_occupation,...,auto_year,policy_duration_days,claim_to_premium_ratio,injury_claim_ratio,property_claim_ratio,vehicle_claim_ratio,is_night_incident,multi_vehicle_flag,actual_fraud,risk_score
56,439,56,IN,250/500,500,1082.49,0,FEMALE,PhD,prof-specialty,...,2014,2291,52.129812,0.000000,0.111111,0.888889,0,1,0,0.998943
915,231,37,OH,250/500,500,920.81,0,FEMALE,High School,sales,...,1997,8464,77.312366,0.000000,0.111111,0.888889,0,0,1,0.998464
914,140,36,OH,500/1000,2000,979.26,0,FEMALE,JD,transport-moving,...,1998,5343,74.341850,0.200000,0.200000,0.600000,1,1,0,0.995344
755,144,35,IL,100/300,500,1427.46,0,MALE,High School,machine-op-inspct,...,1995,7823,61.577908,0.200000,0.100000,0.700000,1,1,0,0.986976
888,381,55,OH,500/1000,500,1459.99,0,FEMALE,MD,other-service,...,2011,8675,41.507134,0.200000,0.100000,0.700000,0,0,0,0.983506
587,463,59,IL,250/500,1000,979.73,0,FEMALE,JD,exec-managerial,...,1999,7498,73.897911,0.100000,0.200000,0.700000,0,1,1,0.975973
748,322,44,IL,100/300,1000,1156.19,0,FEMALE,College,machine-op-inspct,...,2010,3710,42.726541,0.200000,0.100000,0.700000,0,1,0,0.970085
251,111,27,OH,250/500,500,1459.97,5000000,MALE,MD,sales,...,2011,4541,55.377850,0.090909,0.181818,0.727273,1,1,1,0.967961
731,219,43,IN,100/300,1000,1114.29,0,FEMALE,High School,transport-moving,...,2006,4585,59.822847,0.090909,0.090909,0.818182,1,0,0,0.963535
540,138,30,OH,500/1000,500,1093.07,4000000,FEMALE,PhD,other-service,...,2011,4745,76.079300,0.083333,0.166667,0.750000,1,0,0,0.962929


In [15]:
def fraud_capture_rate(df, top_pct):
    n = int(len(df) * top_pct)
    top_df = df.head(n)
    
    total_fraud = df["actual_fraud"].sum()
    captured_fraud = top_df["actual_fraud"].sum()
    
    if total_fraud == 0:
        return 0
    
    return captured_fraud / total_fraud

In [16]:
for pct in [0.10, 0.20, 0.30]:
    capture = fraud_capture_rate(evaluation_df, pct)
    print(f"Top {int(pct*100)}% review captures {capture:.2%} of fraud cases")

Top 10% review captures 20.41% of fraud cases
Top 20% review captures 51.02% of fraud cases
Top 30% review captures 71.43% of fraud cases


In [17]:
joblib.dump(best_pipeline, "../models/fraud_model_pipeline.pkl")
print("Saved to ../models/fraud_model_pipeline.pkl")

Saved to ../models/fraud_model_pipeline.pkl


In [ ]:
import os
os.makedirs("../reports", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

In [ ]:
# Refit best pipeline if needed
best_pipeline.fit(X_train, y_train)

# Create scored evaluation table
risk_scores = best_pipeline.predict_proba(X_test)[:, 1]

evaluation_df = X_test.copy()
evaluation_df["actual_fraud"] = y_test.values
evaluation_df["risk_score"] = risk_scores
evaluation_df = evaluation_df.sort_values(by="risk_score", ascending=False).reset_index(drop=True)

evaluation_df.head()

In [ ]:
def fraud_capture_rate(df, top_pct):
    n = max(1, int(len(df) * top_pct))
    top_df = df.head(n)
    total_fraud = df["actual_fraud"].sum()
    captured_fraud = top_df["actual_fraud"].sum()
    return captured_fraud / total_fraud if total_fraud > 0 else 0

def workload_reduction(top_pct):
    return 1 - top_pct

business_rows = []

for pct in [0.10, 0.20, 0.30]:
    capture = fraud_capture_rate(evaluation_df, pct)
    business_rows.append({
        "review_threshold": f"Top {int(pct*100)}%",
        "claims_reviewed_pct": pct,
        "fraud_capture_rate": round(capture, 4),
        "workload_reduction": round(workload_reduction(pct), 4)
    })

business_metrics_df = pd.DataFrame(business_rows)
business_metrics_df

In [ ]:
business_metrics_df.to_csv("../reports/business_metrics.csv", index=False)
print("Saved to ../reports/business_metrics.csv")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

In [ ]:
y_pred_best = best_pipeline.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig("../reports/figures/confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
y_prob_best = best_pipeline.predict_proba(X_test)[:, 1]

RocCurveDisplay.from_predictions(y_test, y_prob_best)
plt.title(f"ROC Curve - {best_model_name}")
plt.tight_layout()
plt.savefig("../reports/figures/roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## Model Selection Interpretation

The model comparison showed that Logistic Regression achieved the strongest F1 Score, indicating the best balance between precision and recall for this fraud detection problem.

Random Forest achieved the highest ROC-AUC, which suggests stronger ranking ability when the goal is to prioritise suspicious claims by risk score.

For baseline fraud classification and interpretability, Logistic Regression is selected as the preferred model. For claim-ranking and prioritisation analysis, Random Forest remains useful due to its ranking strength.

This distinction is important because the project is designed as a fraud risk prioritisation system, not only a binary classification task.